In [1]:
import librosa
import numpy as np
import pandas as pd

from pathlib import Path

import joblib

In [2]:
PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "rf_advanced_model.pkl"
)

SCALER_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "advanced_scaler.pkl"
)

ENCODER_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "label_encoder.pkl"
)

In [3]:
model = joblib.load(
    MODEL_PATH
)

scaler = joblib.load(
    SCALER_PATH
)

encoder = joblib.load(
    ENCODER_PATH
)

print("Loaded Successfully")

Loaded Successfully


In [4]:
def extract_advanced_features(signal, sr):

    features = {}

    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=13
    )

    for i in range(13):

        features[f"mfcc_{i+1}"] = np.mean(
            mfcc[i]
        )

    chroma = librosa.feature.chroma_stft(
        y=signal,
        sr=sr
    )

    for i in range(12):

        features[f"chroma_{i+1}"] = np.mean(
            chroma[i]
        )

    contrast = librosa.feature.spectral_contrast(
        y=signal,
        sr=sr
    )

    for i in range(7):

        features[f"contrast_{i+1}"] = np.mean(
            contrast[i]
        )

    tonnetz = librosa.feature.tonnetz(
        y=librosa.effects.harmonic(signal),
        sr=sr
    )

    for i in range(6):

        features[f"tonnetz_{i+1}"] = np.mean(
            tonnetz[i]
        )

    return features

In [5]:
def predict_sound(audio_path):

    signal, sr = librosa.load(
        audio_path,
        sr=None
    )

    features = extract_advanced_features(
        signal,
        sr
    )

    feature_df = pd.DataFrame(
        [features]
    )

    scaled = scaler.transform(
        feature_df
    )

    prediction = model.predict(
        scaled
    )

    probabilities = model.predict_proba(
        scaled
    )

    predicted_class = encoder.inverse_transform(
        prediction
    )[0]

    confidence = np.max(
        probabilities
    )

    print(
        "Predicted Class:",
        predicted_class
    )

    print(
        "Confidence:",
        round(confidence * 100, 2),
        "%"
    )

    return predicted_class

In [7]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATASET_PATH = PROJECT_ROOT / "dataset"

for category in DATASET_PATH.iterdir():
    if category.is_dir():
        print(category.name)
        files = list(category.glob("*.wav"))
        if len(files) > 0:
            print("  Sample file:", files[0].name)

ESC-50-master


In [9]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

audio_path = (
    PROJECT_ROOT
    / "dataset"
    / "ESC-50-master"
    / "audio"
)

print(audio_path.exists())

wav_files = list(audio_path.glob("*.wav"))

print("Total wav files:", len(wav_files))

print("First file:", wav_files[0])

True
Total wav files: 2000
First file: C:\Natural_Sound_Statistics_Project\dataset\ESC-50-master\audio\1-100032-A-0.wav


In [10]:
TEST_FILE = wav_files[0]

predict_sound(TEST_FILE)

Predicted Class: water_drops
Confidence: 35.33 %


'water_drops'

In [12]:
import random

TEST_FILE = random.choice(wav_files)

print("Testing:", TEST_FILE.name)

predict_sound(TEST_FILE)

Testing: 5-261439-A-15.wav
Predicted Class: water_drops
Confidence: 67.33 %


'water_drops'

In [13]:
metadata = pd.read_csv(
    PROJECT_ROOT
    / "dataset"
    / "ESC-50-master"
    / "meta"
    / "esc50.csv"
)

metadata.head()

,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


In [14]:
def predict_and_compare(audio_file):

    filename = Path(audio_file).name

    actual = metadata.loc[
        metadata["filename"] == filename,
        "category"
    ].values[0]

    predicted = predict_sound(audio_file)

    print("\nACTUAL CLASS :", actual)
    print("PREDICTED    :", predicted)

    if actual == predicted:
        print("\n✅ CORRECT")
    else:
        print("\n❌ INCORRECT")

In [15]:
TEST_FILE = random.choice(wav_files)

print(TEST_FILE.name)

predict_and_compare(TEST_FILE)

4-161303-A-5.wav
Predicted Class: rooster
Confidence: 39.67 %

ACTUAL CLASS : cat
PREDICTED    : rooster

❌ INCORRECT


In [16]:
import random

correct = 0

for _ in range(20):

    file = random.choice(wav_files)

    filename = file.name

    actual = metadata.loc[
        metadata["filename"] == filename,
        "category"
    ].values[0]

    signal, sr = librosa.load(
        file,
        sr=None
    )

    features = extract_advanced_features(
        signal,
        sr
    )

    feature_df = pd.DataFrame(
        [features]
    )

    scaled = scaler.transform(
        feature_df
    )

    pred = encoder.inverse_transform(
        model.predict(scaled)
    )[0]

    if pred == actual:
        correct += 1

print(
    "Accuracy on 20 random files:",
    correct,
    "/20 =",
    round(correct/20*100,2),
    "%"
)

Accuracy on 20 random files: 3 /20 = 15.0 %


In [17]:
trained_classes = [
    "chirping_birds",
    "crackling_fire",
    "crickets",
    "crow",
    "frog",
    "insects",
    "pouring_water",
    "rain",
    "rooster",
    "sea_waves",
    "thunderstorm",
    "water_drops",
    "wind"
]

In [18]:
valid_metadata = metadata[
    metadata["category"].isin(trained_classes)
]

print(valid_metadata.shape)

(520, 7)


In [19]:
valid_wav_files = []

for filename in valid_metadata["filename"]:
    valid_wav_files.append(
        audio_path / filename
    )

print(len(valid_wav_files))

520


In [21]:
import random

correct = 0

for _ in range(20):

    file = random.choice(valid_wav_files)

    filename = file.name

    actual = metadata.loc[
        metadata["filename"] == filename,
        "category"
    ].values[0]

    signal, sr = librosa.load(
        file,
        sr=None
    )

    features = extract_advanced_features(
        signal,
        sr
    )

    feature_df = pd.DataFrame([features])

    scaled = scaler.transform(feature_df)

    pred = encoder.inverse_transform(
        model.predict(scaled)
    )[0]

    if pred == actual:
        correct += 1

print(
    f"Accuracy on 20 valid files: {correct}/20 = {correct/20*100:.2f}%"
)

Accuracy on 20 valid files: 20/20 = 100.00%


In [22]:
from sklearn.metrics import accuracy_score

predictions = []

for file in valid_wav_files:

    signal, sr = librosa.load(
        file,
        sr=None
    )

    features = extract_advanced_features(
        signal,
        sr
    )

    feature_df = pd.DataFrame([features])

    scaled = scaler.transform(feature_df)

    pred = model.predict(scaled)[0]

    predictions.append(pred)

true_labels = encoder.transform(
    valid_metadata["category"]
)

acc = accuracy_score(
    true_labels,
    predictions
)

print("Full Dataset Accuracy:", round(acc*100, 2), "%")

C:\Natural_Sound_Statistics_Project\venv\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
C:\Natural_Sound_Statistics_Project\venv\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
C:\Natural_Sound_Statistics_Project\venv\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
C:\Natural_Sound_Statistics_Project\venv\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
C:\Natural_Sound_Statistics_Project\venv\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Full Dataset Accuracy: 94.42 %
